<a href="https://colab.research.google.com/github/tanusiya1510/DAA/blob/main/E7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
def matrix_chain_memo(dims):
    """
    Matrix Chain Multiplication using Memoization (Top-Down DP)
    dims: list of dimensions, matrix i has dims[i-1] x dims[i]
    Time: O(n^3), Space: O(n^2)
    """
    n = len(dims) - 1

    # Cache table
    m = [[-1] * (n + 1) for _ in range(n + 1)]
    s = [[0] * (n + 1) for _ in range(n + 1)]

    def solve(i, j):
        # Base case
        if i == j:
            return 0

        # Return cached value
        if m[i][j] != -1:
            return m[i][j]

        m[i][j] = float('inf')

        # Try all possible splits
        for k in range(i, j):
            cost = (
                solve(i, k)
                + solve(k + 1, j)
                + dims[i - 1] * dims[k] * dims[j]
            )

            if cost < m[i][j]:
                m[i][j] = cost
                s[i][j] = k

        return m[i][j]

    solve(1, n)

    return m, s


def matrix_chain_bottom_up(dims):
    """
    Matrix Chain Multiplication using Bottom-Up DP
    dims: list of dimensions
    Time: O(n^3), Space: O(n^2)
    """
    n = len(dims) - 1

    m = [[0] * (n + 1) for _ in range(n + 1)]
    s = [[0] * (n + 1) for _ in range(n + 1)]

    for l in range(2, n + 1):
        for i in range(1, n - l + 2):
            j = i + l - 1
            m[i][j] = float('inf')

            for k in range(i, j):
                cost = (
                    m[i][k]
                    + m[k + 1][j]
                    + dims[i - 1] * dims[k] * dims[j]
                )

                if cost < m[i][j]:
                    m[i][j] = cost
                    s[i][j] = k

    return m, s


def print_optimal_parens(s, i, j):
    if i == j:
        return f'A{i}'

    k = s[i][j]

    left = print_optimal_parens(s, i, k)
    right = print_optimal_parens(s, k + 1, j)

    return f'({left} x {right})'


def print_dp_table(m, n):
    print("\nDP Cost Table m[i][j]:")

    print(f'{"":>6}', end='')

    for j in range(1, n + 1):
        print(f'A{j:>10}', end='')

    print()

    for i in range(1, n + 1):
        print(f'A{i:<5}', end='')

        for j in range(1, n + 1):
            if j < i:
                print(f'{"---":>11}', end='')
            else:
                print(f'{m[i][j]:>11}', end='')

        print()


# --- Chain of 6 Matrices ---
# A1(10x30), A2(30x5), A3(5x60),
# A4(60x10), A5(10x20), A6(20x15)

dims = [10, 30, 5, 60, 10, 20, 15]

names = ["A1", "A2", "A3", "A4", "A5", "A6"]

n = len(dims) - 1


print("Matrix Dimensions:")

for i in range(n):
    print(f' {names[i]}: {dims[i]} x {dims[i + 1]}')


# --- Top-Down Memoized DP ---
memo, memo_split = matrix_chain_memo(dims)


# --- Bottom-Up DP ---
bottom, bottom_split = matrix_chain_bottom_up(dims)


print("\nMemoized Top-Down Result:")
print(f"Minimum scalar multiplications: {memo[1][n]}")
print(f"Optimal parenthesization: {print_optimal_parens(memo_split, 1, n)}")


print("\nBottom-Up DP Result:")
print(f"Minimum scalar multiplications: {bottom[1][n]}")
print(f"Optimal parenthesization: {print_optimal_parens(bottom_split, 1, n)}")


# --- Verification ---
print("\nVerification:")

if memo[1][n] == bottom[1][n]:
    print("Both approaches give the same minimum cost.")
else:
    print("Results are different.")


print_dp_table(bottom, n)

Matrix Dimensions:
 A1: 10 x 30
 A2: 30 x 5
 A3: 5 x 60
 A4: 60 x 10
 A5: 10 x 20
 A6: 20 x 15

Memoized Top-Down Result:
Minimum scalar multiplications: 7750
Optimal parenthesization: ((A1 x A2) x (((A3 x A4) x A5) x A6))

Bottom-Up DP Result:
Minimum scalar multiplications: 7750
Optimal parenthesization: ((A1 x A2) x (((A3 x A4) x A5) x A6))

Verification:
Both approaches give the same minimum cost.

DP Cost Table m[i][j]:
      A         1A         2A         3A         4A         5A         6
A1              0       1500       4500       5000       6500       7750
A2            ---          0       9000       4500       7000       7750
A3            ---        ---          0       3000       4000       5500
A4            ---        ---        ---          0      12000      12000
A5            ---        ---        ---        ---          0       3000
A6            ---        ---        ---        ---        ---          0


In [2]:
import time


def solve_n_queens_bitmask(n):
    """
    N-Queens Problem using Backtracking with Bitmask
    Bit manipulation is used for tracking columns and diagonals.
    is_safe check is reduced to O(1).
    Time: O(N!), Space: O(N)
    """

    solutions = []
    backtrack_count = 0
    board = [-1] * n

    def backtrack(row, cols, diag1, diag2):
        nonlocal backtrack_count

        # All queens placed
        if row == n:
            solutions.append(board.copy())
            return

        # Available positions using bit operations
        available = ((1 << n) - 1) & ~(cols | diag1 | diag2)

        while available:
            # Get the rightmost available position
            position = available & -available

            # Remove that position
            available -= position

            # Find column number
            col = position.bit_length() - 1

            board[row] = col

            backtrack_count += 1

            # Move to next row
            backtrack(
                row + 1,
                cols | position,
                (diag1 | position) << 1,
                (diag2 | position) >> 1
            )

    backtrack(0, 0, 0, 0)

    return solutions, backtrack_count


def print_solution(solution):
    n = len(solution)

    for row in range(n):
        line = ""

        for col in range(n):
            if solution[row] == col:
                line += " Q "
            else:
                line += " . "

        print(line)

    print()


# --- Demonstration for Small N ---
n = 4

solutions, count = solve_n_queens_bitmask(n)

print(f"N = {n}")
print(f"Number of Solutions: {len(solutions)}")
print(f"Total Backtracks: {count}")

print("\nOne Solution:")
print_solution(solutions[0])


# --- Performance Comparison for N = 12 ---
n = 12

start = time.time()

solutions, backtracks = solve_n_queens_bitmask(n)

end = time.time()

print(f"\nPerformance Analysis for N = {n}")
print("-" * 50)

print(f"Total Solutions Found : {len(solutions)}")
print(f"Total Backtracks      : {backtracks}")
print(f"Execution Time        : {end - start:.6f} seconds")


print("\nAnalysis:")
print("Bitmask representation avoids checking every column and diagonal")
print("using loops in is_safe().")
print("Each safety check is performed using bitwise AND and OR operations.")
print("Therefore, is_safe() complexity is reduced from O(N) to O(1).")

N = 4
Number of Solutions: 2
Total Backtracks: 16

One Solution:
 .  Q  .  . 
 .  .  .  Q 
 Q  .  .  . 
 .  .  Q  . 


Performance Analysis for N = 12
--------------------------------------------------
Total Solutions Found : 14200
Total Backtracks      : 856188
Execution Time        : 0.550481 seconds

Analysis:
Bitmask representation avoids checking every column and diagonal
using loops in is_safe().
Each safety check is performed using bitwise AND and OR operations.
Therefore, is_safe() complexity is reduced from O(N) to O(1).


In [3]:
def knight_tour(n):
    """
    Knight's Tour Problem using Backtracking
    Finds a Hamiltonian Path where the knight visits every cell exactly once.
    Time: O(8^(N^2)), Space: O(N^2)
    """

    board = [[-1 for _ in range(n)] for _ in range(n)]

    # Knight movement possibilities
    moves = [
        (2, 1), (1, 2), (-1, 2), (-2, 1),
        (-2, -1), (-1, -2), (1, -2), (2, -1)
    ]

    board[0][0] = 0   # Starting position (0,0)

    move_count = 0

    def is_safe(x, y):
        return (
            0 <= x < n and
            0 <= y < n and
            board[x][y] == -1
        )

    def solve(x, y, move_number):
        nonlocal move_count

        # All cells visited
        if move_number == n * n:
            return True

        for dx, dy in moves:
            next_x = x + dx
            next_y = y + dy

            if is_safe(next_x, next_y):

                board[next_x][next_y] = move_number
                move_count += 1

                if solve(next_x, next_y, move_number + 1):
                    return True

                # Backtrack
                board[next_x][next_y] = -1

        return False

    if solve(0, 0, 1):
        return board, move_count

    return None, move_count


def print_board(board):
    n = len(board)

    print("\nKnight's Tour Solution:")
    print("-" * 50)

    for i in range(n):
        for j in range(n):
            print(f'{board[i][j]:>3}', end=" ")
        print()


# --- Demonstration for 8x8 Chessboard ---

n = 8

solution, backtracks = knight_tour(n)

if solution:
    print(f"Knight's Tour found for {n}x{n} chessboard")
    print(f"Starting Position: (0,0)")
    print_board(solution)
    print(f"\nTotal Backtracks: {backtracks}")

else:
    print("No solution exists.")


print("\nAnalysis:")
print("Knight's Tour is solved using Backtracking.")
print("Each move is checked and invalid paths are undone.")
print("Time Complexity: O(8^(N^2)) in worst case.")
print("Space Complexity: O(N^2) for storing the chessboard.")

Knight's Tour found for 8x8 chessboard
Starting Position: (0,0)

Knight's Tour Solution:
--------------------------------------------------
  0  59  38  33  30  17   8  63 
 37  34  31  60   9  62  29  16 
 58   1  36  39  32  27  18   7 
 35  48  41  26  61  10  15  28 
 42  57   2  49  40  23   6  19 
 47  50  45  54  25  20  11  14 
 56  43  52   3  22  13  24   5 
 51  46  55  44  53   4  21  12 

Total Backtracks: 8250732

Analysis:
Knight's Tour is solved using Backtracking.
Each move is checked and invalid paths are undone.
Time Complexity: O(8^(N^2)) in worst case.
Space Complexity: O(N^2) for storing the chessboard.
